# 02｜State 模型：真实配置、集合张量与权重迁移

**目标：** 读真实发布配置，区分 SE/ST、HVG/全基因、训练/推理；用可运行的 NumPy 小例子理解 cell token、集合注意力、Energy loss 和基因坐标迁移。

可独立从头运行，CPU 即可，不下载权重。配置快照是固定 revision 的官方 YAML；模型数值演示是**随机初始化的教学模型，不是官方 State，也不具备真实扰动预测能力**。这里不把玩具结果当成生物证据。

前一课：[01｜真实数据](01_vc2026_data.ipynb)；完整原理与命令：[L3-04 State 微调教程](../docs/lessons/L3-04-State微调实战与算力预算.md)。

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display, Markdown

# 从仓库根目录或 notebook/ 启动都可以；不写死某台机器的绝对路径。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').exists()
             and (p / 'docs/Official-website').is_dir()), None)
if ROOT is None:
    raise RuntimeError('请在 virtual-cell-2026 仓库内启动 notebook')
DATA = ROOT / 'data/vcc2026-validation/controls'
if not (DATA / 'manifest.json').exists():
    raise FileNotFoundError(f'缺少 {DATA}；请先放好官方 controls 数据包')
OUT = ROOT / 'output/notebook-learning/02-model'
OUT.mkdir(parents=True, exist_ok=True)
manifest = json.loads((DATA / 'manifest.json').read_text())
genes = pd.read_csv(DATA / 'gene_names.csv')['gene_name'].astype(str).tolist()
targets = pd.read_csv(DATA / 'pert_counts.csv')['target_gene'].astype(str).tolist()
contexts = manifest['contexts']
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'],
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.dpi': 110,
    'svg.fonttype': 'none', 'pdf.fonttype': 42,
})
COLORS = {'A': '#167B72', 'B': '#3E71AD', 'C': '#DC654F'}
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 12)
def show_figure(fig, name):
    # 完整可再生成图只写入 Git 忽略目录；notebook 内嵌适合阅读的预览。
    fig.savefig(OUT / f'{name}.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT / f'{name}.svg', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)
print('Python:', sys.version.split()[0], '| 数据目录:', DATA.relative_to(ROOT))
print('本课输出目录:', OUT.relative_to(ROOT))

Python: 3.13.14 | 数据目录: data/vcc2026-validation/controls
本课输出目录: output/notebook-learning/02-model


## 单元一：我们下载的是哪一个 State

**SE（State Embedding，状态嵌入）** 把细胞表达压缩成表示；**ST（State Transition，状态转移）** 用对照细胞和扰动条件预测变化。一个 SE 模型不自动等于一个扰动预测器。

**HVG（highly variable genes，高变基因）** 是一部分变化较明显的基因。`ST-HVG-Replogle` 使用其中 2,000 个，而 `st-x-replogle-full` 示例仅有 6,546 个；名称里的 full 不是本赛完整 18,533 个基因。

下面读取**未改写的官方 YAML**。源 URL、revision、获取日期与 SHA-256 位于 `assets/state/sources.json`。其中作者服务器路径只表示来源，不是本机可运行路径。我们没有加载 checkpoint 张量或核实其所有训练样本。

In [2]:
import yaml
ASSETS = ROOT / 'notebook/assets/state'
source_manifest = json.loads((ASSETS / 'sources.json').read_text())
for record in source_manifest:
    payload = (ASSETS / record['file']).read_bytes()
    assert hashlib.sha256(payload).hexdigest() == record['sha256']
configs = {key: yaml.safe_load((ASSETS / f'{key}_config.yaml').read_text())
           for key in ['hvg_jurkat', 'full_k562']}
hparams = {key: yaml.safe_load((ASSETS / f'{key}_hparams.yaml').read_text())
           for key in configs}
display(pd.DataFrame([{'file': r['file'], 'bytes': r['bytes'], 'revision': r['revision'][:12]}
                      for r in source_manifest]))
rows = []
for key, cfg in configs.items():
    hp, model_cfg = hparams[key], cfg['model']['kwargs']
    backbone = model_cfg['transformer_backbone_kwargs']
    rows.append({'asset': key, 'input_G': hp['input_dim'], 'output_G': hp['output_dim'],
                 'hidden': hp['hidden_dim'], 'sets_of_cells': model_cfg['cell_set_len'],
                 'layers': backbone['num_hidden_layers'],
                 'heads': backbone['num_attention_heads'], 'head_dim': backbone['head_dim'],
                 'pert_dim': hp['pert_dim'], 'batch_encoder': model_cfg['batch_encoder']})
display(pd.DataFrame(rows))
display(Markdown('官方配置入口：' + ' · '.join(
    f"[{r['file']}]({r['url']})" for r in source_manifest if r['file'].endswith('_config.yaml'))))

,file,bytes,revision
0,hvg_jurkat_config.yaml,2766,bb6a9562cbbf
1,hvg_jurkat_hparams.yaml,18468,bb6a9562cbbf
2,full_k562_config.yaml,2736,48ad5f70215a
3,full_k562_hparams.yaml,56363,48ad5f70215a


,asset,input_G,output_G,hidden,sets_of_cells,layers,heads,head_dim,pert_dim,batch_encoder
0,hvg_jurkat,2000,2000,328,64,8,12,64,2024,True
1,full_k562,6546,6546,328,64,8,12,64,2024,True


官方配置入口：[hvg_jurkat_config.yaml](https://huggingface.co/arcinstitute/ST-HVG-Replogle/resolve/bb6a9562cbbf1fd152df14cc53b4cc7517c77175/zeroshot/jurkat/config.yaml) · [full_k562_config.yaml](https://huggingface.co/arcinstitute/st-x-replogle-full/resolve/48ad5f70215ab4c58caa5a68e77d837601d29d35/k562_0.99/config.yaml)

**读表方法：** hidden=328 是细胞隐表示宽度，set=64 是一组中的细胞数，pert_dim=2024 是旧靶点向量维数。这里不是“一个基因一个 token”，也不是 2,024 种细胞。

官方配置显式设置 `12 heads × head_dim 64`，不能按一般经验强行令 328 被 12 整除。当前源码默认 State 是 hidden 768、set 512，与这两组发布权重不同。微调必须继承父模型结构，而不是只传一个 checkpoint 路径。

首先把旧基因轴与比赛轴比较。**这只核验模型读出坐标，不是训练目标覆盖**；目标是否训练过要检查 perturbation mapping 与预训练来源。

In [3]:
coverage = []
for key, hp in hparams.items():
    old = hp['gene_names']
    assert len(old) == len(set(old)) == hp['output_dim']
    common = set(old) & set(genes)
    coverage.append({'asset': key, 'checkpoint_genes': len(old),
                     'shared_with_2026': len(common), 'old_genes_not_in_2026': len(set(old)-set(genes)),
                     '2026_genes_outside_old_head': len(set(genes)-set(old))})
display(pd.DataFrame(coverage))
display(pd.DataFrame({'old_HVG_axis_first_8': hparams['hvg_jurkat']['gene_names'][:8],
                      '2026_axis_first_8': genes[:8]}))
print('未下载 pert_onehot_map.pt；不根据 pert_dim 猜测 300 个目标的覆盖率。')

,asset,checkpoint_genes,shared_with_2026,old_genes_not_in_2026,2026_genes_outside_old_head
0,hvg_jurkat,2000,1877,123,16656
1,full_k562,6546,6197,349,12336


,old_HVG_axis_first_8,2026_axis_first_8
0,HES4,TSPAN6
1,ISG15,TNMD
2,MIB2,DPM1
3,CEP104,SCYL3
4,ACOT7,C1orf112
5,VAMP3,FGR
6,RERE,CFH
7,ENO1,FUCA2


未下载 pert_onehot_map.pt；不根据 pert_dim 猜测 300 个目标的覆盖率。


### 用真实覆盖结果修正实施预期

本次本地文件的**原始符号逐字匹配**结果是：旧 HVG 的 2,000 个基因中匹配 1,877 个、123 个未直接匹配；旧 full 的 6,546 个中匹配 6,197 个、349 个未直接匹配。以你重新运行时的上表为准。

未匹配不一定等于生物学上未测：旧符号、别名或 ID 版本也会造成不匹配。先建立有来源的名称映射，再确定真正的缺测集合。若仍缺父模型输入基因，就不能直接把官方 NTC 塞入旧输入层，或补零后宣称无损兼容；需要明确输入适配、缺失标记/迁移和相应验证。

因此，“先复用权重”表示先审计可复用部分，并不保证完整旧输入输出接口可以直接使用。这个覆盖实验本身没有检查 300 个扰动目标的训练暴露。

## 单元二：NTC 如何进入集合模型

一个训练样本的生物对象是：同背景、同扰动条件的一组细胞。输入 NTC 来自匹配背景/实验批次，监督是相同条件下另一组真实扰动细胞。测序破坏细胞，不能自然配对“同一个细胞扰动前后”的两行。

维度写作 `B × S × G`：B 个集合，每集合 S 个细胞，每细胞 G 个基因。每个细胞一个 token。靶点特征经投影后加到集合内每个细胞的隐表示。注意力可以利用同组其他细胞的信息；相同基因扰动在不同背景中可能有不同结果。

下面读取真实 A/B NTC，各取 16 行，**只为展示张量操作，截取前 48 个基因**。这不是 HVG 选择、训练数据筛选或统计结论；归一化分母仍由全部 18,533 个基因计算。教学尺寸与发布尺寸分别列出，不相互替代。

In [4]:
S, G, H, D = 16, 48, 16, 8
demo_genes = genes[:G]
input_sets = []
for context in ['A', 'B']:
    a = ad.read_h5ad(DATA / f'context_{context}.h5ad', backed='r')
    try:
        counts = a.X[:S].tocsr().astype(np.float64)
    finally:
        a.file.close()
    totals = np.asarray(counts.sum(axis=1)).ravel()
    assert (totals > 0).all()
    normalized = counts.multiply(10000 / totals[:, None]).tocsr()
    input_sets.append(np.log1p(normalized[:, :G].toarray()))
x = np.stack(input_sets)
B = x.shape[0]
display(pd.DataFrame([{'tensor': 'NTC 表达 x', 'shape': str(x.shape), 'meaning': 'B×S×G'},
                      {'tensor': '投影后细胞 token', 'shape': str((B,S,H)), 'meaning': 'B×S×H'},
                      {'tensor': '每组靶点特征', 'shape': str((B,D)), 'meaning': 'B×D'}]))

,tensor,shape,meaning
0,NTC 表达 x,"(2, 16, 48)",B×S×G
1,投影后细胞 token,"(2, 16, 16)",B×S×H
2,每组靶点特征,"(2, 8)",B×D


### one-hot 与 ESM2 各自提供什么

**one-hot（独热编码）**给每个已知目标一个固定坐标；改动词表顺序会改变坐标含义。**ESM2**把蛋白质氨基酸序列编码成连续向量，让新目标能被模型接收；它不是一个保证准确的生物响应数据库，也不能直接表示所有非蛋白编码目标。

下方 `toy_target_features` 是随机演示向量，**不是 ESM2**。正式训练需下载来源明确的特征，统一序列/层/pooling 方法，并检查全部目标及 non-targeting 的覆盖。

In [5]:
rng = np.random.default_rng(42)
toy_target_features = {name: rng.normal(size=D) for name in ['ADNP', 'ACLY']}
toy_target_features['non-targeting'] = np.zeros(D)
example_target = 'ADNP'  # 可换为 ACLY，再从这里运行后续单元
features = np.stack([toy_target_features[example_target]] * B)
W_cell = rng.normal(scale=0.1, size=(G,H))
W_target = rng.normal(scale=0.1, size=(D,H))
Wq, Wk, Wv = [rng.normal(scale=0.2, size=(H,H)) for _ in range(3)]
W_out = rng.normal(scale=0.03, size=(H,G))
print('教学目标:', example_target, '| 特征 shape:', features.shape, '| 向量来源: 随机数学示例')

教学目标: ADNP | 特征 shape: (2, 8) | 向量来源: 随机数学示例


### 一个可以看穿每一步的集合注意力

1. `x @ W_cell` 将表达投影为细胞 token。
2. `features @ W_target` 将靶点条件加到每个 token。
3. `Q @ K.T` 比较同一组细胞；softmax 得到关注权重。
4. 加权汇总 V、投影回基因维，输出一组表达。

这是单头、单层 NumPy 教学模型，省略官方多头、8 层、归一化和不同残差路径。它只解释“细胞集合如何流经网络”，不会从随机参数产生有意义的 CRISPRi 预测。

In [6]:
def softmax(a):
    shifted = a - a.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)

def toy_forward(x, target_features):
    hidden = x @ W_cell + (target_features @ W_target)[:, None, :]
    q, k, v = hidden @ Wq, hidden @ Wk, hidden @ Wv
    attention = softmax((q @ k.swapaxes(-1,-2)) / np.sqrt(H))
    updated = hidden + attention @ v
    prediction = np.maximum(x + updated @ W_out, 0)
    return prediction, attention

toy_prediction, attention = toy_forward(x, features)
assert toy_prediction.shape == x.shape
assert np.allclose(attention.sum(axis=-1), 1)
display(pd.DataFrame({'stage': ['input', 'target projection', 'attention', 'prediction'],
                      'shape': [str(x.shape), str((B,H)), str(attention.shape), str(toy_prediction.shape)]}))
display(pd.DataFrame(attention[0,:5,:5]).round(3))

,stage,shape
0,input,"(2, 16, 48)"
1,target projection,"(2, 16)"
2,attention,"(2, 16, 16)"
3,prediction,"(2, 16, 48)"


,0,1,2,3,4
0,0.063,0.067,0.067,0.064,0.058
1,0.065,0.068,0.067,0.063,0.057
2,0.066,0.067,0.066,0.067,0.056
3,0.071,0.064,0.066,0.065,0.055
4,0.065,0.067,0.067,0.066,0.056


### 把细胞行重新排序，会不会改变生物问题

细胞集合没有自然句子顺序。一个不带位置编码的集合映射，应在输入细胞顺序变化时，让输出对应重排，集合本身不变。这叫**置换等变（permutation equivariance）**。

下面是一个真正运行的性质检查，而不是一张预制模型示意图。它只验证这个教学 forward；官方实现的 padding、分组和推理切片仍需单独测试。

In [7]:
permutation = rng.permutation(S)
reordered_prediction, _ = toy_forward(x[:, permutation], features)
assert np.allclose(reordered_prediction, toy_prediction[:, permutation], atol=1e-10)
print('通过：仅重排同集合细胞，输出也仅按相同顺序重排。')
other_features = np.stack([toy_target_features['ACLY']] * B)
other_prediction, _ = toy_forward(x, other_features)
print('改变条件是否改变演示输出:', not np.allclose(other_prediction, toy_prediction))
print('这证明条件进入了计算，不证明新条件的生物效果正确。')

通过：仅重排同集合细胞，输出也仅按相同顺序重排。
改变条件是否改变演示输出: True
这证明条件进入了计算，不证明新条件的生物效果正确。


## 单元三：为什么用集合损失，而不是强迫细胞逐行配对

**Energy distance（能量距离）**比较两个集合。这里演示常用形式：

`2 × 预测与真值的平均距离 − 预测内部平均距离 − 真值内部平均距离`。

距离采用欧氏距离；原版 State 的具体缩放与实现由 GeomLoss 决定。集合距离对行顺序不敏感；逐行 MSE 则暗示了并不存在的配对。注意力让细胞共享上下文，集合损失让监督不依赖任意排序，这两点职责不同。

下面的人造 `synthetic_truth` 只是置换实验，并非官方 A/B 扰动标签。全部 16 个点和 48 个维度进入计算，不报告任何比赛指标。

In [8]:
from scipy.spatial.distance import cdist

def energy_distance(a, b):
    return 2*cdist(a,b).mean() - cdist(a,a).mean() - cdist(b,b).mean()

synthetic_truth = toy_prediction[0].copy()  # 人工设为同一集合
shuffled_truth = synthetic_truth[permutation]
comparison = pd.DataFrame([
    {'truth_order': 'same', 'paired_MSE': np.mean((toy_prediction[0]-synthetic_truth)**2),
     'energy_demo': energy_distance(toy_prediction[0],synthetic_truth)},
    {'truth_order': 'shuffled', 'paired_MSE': np.mean((toy_prediction[0]-shuffled_truth)**2),
     'energy_demo': energy_distance(toy_prediction[0],shuffled_truth)},
])
display(comparison)
assert np.isclose(comparison.energy_demo.iloc[0], comparison.energy_demo.iloc[1], atol=1e-10)

,truth_order,paired_MSE,energy_demo
0,same,0.000000,0.0
1,shuffled,0.210114,0.0


**诊断题：**上述置换测试和条件敏感性都通过了，能否说教学模型学会了跨背景扰动？

<details><summary>参考答案</summary>不能。这里只检查了输入输出性质，参数没有在真实公共扰动标签上训练。性能需要无泄漏的背景/靶点留出，并在计数输出后测六指标。</details>

## 单元四：微调并不只是 load 一个文件

把旧权重想成“带坐标含义的矩阵”。同样的 shape 可能对应不同基因顺序。下面用三基因线性层演示一个不会抛异常、却会改变结果的错误；名称来自例子，不解释其生物作用。

In [9]:
old_genes = ['ADNP','ACLY','TP53']
new_genes = ['TP53','ADNP','ACLY']
old_expression = np.array([2., 5., 1.])
old_weight = np.array([[1.,2.],[3.,4.],[5.,6.]])  # 输入基因×hidden
new_expression = old_expression[[old_genes.index(g) for g in new_genes]]
wrong = new_expression @ old_weight
name_aligned_weight = np.stack([old_weight[old_genes.index(g)] for g in new_genes])
correct = new_expression @ name_aligned_weight
expected = old_expression @ old_weight
display(pd.DataFrame([expected, wrong, correct], index=['old axis','same shape, wrong meaning','name aligned'],
                      columns=['hidden_1','hidden_2']))
assert not np.allclose(wrong, expected)
assert np.allclose(correct, expected)

,hidden_1,hidden_2
old axis,22.0,30.0
"same shape, wrong meaning",32.0,40.0
name aligned,22.0,30.0


实际 ST 的输入 Linear 权重是 PyTorch 的 `[hidden, input_G]`，因此要按基因拷贝**列**；输出 Linear 是 `[output_G, hidden]`，按基因拷贝**行**。上面的 NumPy 为了书写 `x @ W` 使用了转置约定。

缺测基因也要分清：公开数据没有该列，不是测到了零。全基因训练需要测量 mask；旧 2,000 HVG 如果有缺测，同样需要处理，不能默默补零监督。

| 需求 | 原生行为 | 我们需要补什么 |
|---|---|---|
| 初始化微调 | `model.kwargs.init_from` 按键和 shape 过滤后加载 | 逐模块继承比例和坐标验证 |
| 继续同一 run | 存在 `last.ckpt` 时优先 resume | 新实验用新目录，防止误恢复旧 optimizer |
| one-hot → ESM2 | 维度不同的层被跳过；同维语义变化不会自动发现 | 显式重建靶点层并训练 |
| HVG → 全基因 | 输入/输出层变形；all 路径还新增基因混合层 | 名称映射、mask、输出与计数适配 |
| 只训练新接口 | `freeze_pert_backbone` 还会冻住输出层 | 显式指定 requires_grad/optimizer 参数组 |
| LoRA | 注入可能改变 key 名称 | 先确认基础权重继承，再验证适配顺序 |

不是所有额外工作都已由原生开关完成。详见 [检查点审计](../docs/research/state-checkpoint-finetuning-audit.md)。

In [10]:
def gene_mixing_parameters(g):
    hidden = g // 8
    return g*hidden + hidden + hidden*g + g

display(pd.DataFrame([{'G': g, 'G_to_Gdiv8_to_G_parameters': gene_mixing_parameters(g),
                       'FP32_weights_MiB': gene_mixing_parameters(g)*4/2**20}
                      for g in [6546,18533]]))
print('HVG gene 路线没有这一 all 路径层；表中只算该模块，不是整个模型参数。')
print('HVG -> all 还会改变残差位置，不能把扩维理解成完全保持旧函数。')

,G,G_to_Gdiv8_to_G_parameters,FP32_weights_MiB
0,6546,10716620,40.880661
1,18533,85865705,327.551670


HVG gene 路线没有这一 all 路径层；表中只算该模块，不是整个模型参数。
HVG -> all 还会改变残差位置，不能把扩维理解成完全保持旧函数。


## 单元五：这个模型到哪里才能学到真实响应

公共 CRISPRi 数据提供“对照 → 扰动后”的监督；A/B/C 仅提供目标背景。零样本限制的是目标背景的扰动标签，不禁止在外部背景训练。

Replogle 的 K562/RPE1 加 Nadig 的 HepG2/Jurkat 是起步数据。H1 是额外的 Flex 背景开发集：若要称“未见背景”，从父权重到微调都不能使用 H1 的扰动结果。只从本次微调集删除 Jurkat，也不能擦除父权重已经看过它的历史。

论文和公开权重没有替你解决新面板覆盖、数据噪声、原始计数和在线分数。下一课将这些环节拆成可检查产物，并建立资源预算。

- 下一份：[03｜微调、计数与资源](03_finetuning_counts_and_budget.ipynb)
- 官方权重：[ST-HVG-Replogle](https://huggingface.co/arcinstitute/ST-HVG-Replogle)、[st-x-replogle-full](https://huggingface.co/arcinstitute/st-x-replogle-full)
- 固定代码：[State 9bbfe78a](https://github.com/ArcInstitute/state/tree/9bbfe78a434a55205e4de834e1ea99f85f7a3add)
- 论文：[State 正式 DOI](https://doi.org/10.1016/j.cell.2026.07.052)；本文方法解释基于已核验代码与既有教程，不声称重读正式全文。